# Notebook 01 — Synthetic Data Simulation

**Goal:** Generate ~100K synthetic users with a *known* heterogeneous treatment effect (ITE) function.  
This lets us later validate our uplift model against the true answer — something you can't do on real data.

## Design Choices
| Segment | True ITE (conversion lift) | Rationale |
|---|---|---|
| New mobile users (tenure < 90d) | +8 to +12pp | High receptivity to engagement nudges |
| New desktop users (tenure < 90d) | +3 to +5pp | Positive but weaker |
| Mid-tenure mobile (90–365d) | +1 to +3pp | Marginal |
| Long-tenure desktop (> 365d) | -1 to +1pp | Near-zero, possible annoyance |
| Tablet users | -2 to +0pp | Slight negative (novelty fatigue) |

This heterogeneity is *deliberately designed* to test whether our uplift model can recover the correct segment ranking.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED = 42
rng  = np.random.default_rng(SEED)

# ── Style ────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#0f1117',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#e6edf3',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'font.size':        11,
})

PALETTE = ['#58a6ff', '#3fb950', '#f78166', '#d2a8ff', '#ffa657']

Path('data').mkdir(exist_ok=True)
Path('figures').mkdir(exist_ok=True)

print('Libraries loaded ✓')

## 1. Generate User Covariates

In [ ]:
N = 100_000

# ── Covariates ───────────────────────────────────────────────────────────────
age = np.clip(rng.normal(35, 12, N), 18, 70).astype(int)

# Tenure: most users are new, heavy tail of long-term users
tenure_days = np.clip(
    rng.exponential(scale=180, size=N), 0, 1825
).astype(int)

device = rng.choice(['mobile', 'desktop', 'tablet'],
                    p=[0.55, 0.35, 0.10], size=N)

# Past spend: log-normal, right-skewed
past_spend = np.clip(rng.lognormal(mean=1.5, sigma=1.2, size=N), 0, 500)

# Engagement score 0–1 (Beta distribution — most users are low-engagement)
engagement_score = rng.beta(a=1.5, b=4.0, size=N)

df = pd.DataFrame({
    'user_id':          np.arange(N),
    'age':              age,
    'tenure_days':      tenure_days,
    'device':           device,
    'past_spend':       past_spend.round(2),
    'engagement_score': engagement_score.round(4),
})

print(f'Dataset shape: {df.shape}')
df.head()

## 2. Define the True Heterogeneous Treatment Effect (ITE)

In [ ]:
def true_ite(row):
    """
    Ground-truth individual treatment effect function.
    Returns expected lift in conversion probability (additive, on probability scale).
    
    This is the function our uplift model will try to recover.
    """
    device   = row['device']
    tenure   = row['tenure_days']
    eng      = row['engagement_score']

    # ── Tablet: slight negative effect (novelty fatigue) ─────────────────────
    if device == 'tablet':
        return -0.01 + eng * 0.01   # ranges from ~-1% to ~0%

    # ── New users (< 90 days) ────────────────────────────────────────────────
    if tenure < 90:
        if device == 'mobile':
            return 0.08 + eng * 0.04   # +8% to +12%
        else:  # desktop
            return 0.03 + eng * 0.02   # +3% to +5%

    # ── Mid-tenure (90–365 days) ──────────────────────────────────────────────
    if tenure < 365:
        if device == 'mobile':
            return 0.01 + eng * 0.02   # +1% to +3%
        else:
            return 0.005 + eng * 0.01  # +0.5% to +1.5%

    # ── Long-tenure (> 365 days) ──────────────────────────────────────────────
    if device == 'mobile':
        return eng * 0.01 - 0.005      # small, ~zero
    else:  # desktop
        return eng * 0.005 - 0.01      # slightly negative


df['ite_true'] = df.apply(true_ite, axis=1).round(6)
print(f'ITE range: [{df.ite_true.min():.4f}, {df.ite_true.max():.4f}]')
df[['ite_true']].describe()

## 3. Generate Potential Outcomes Y(0) and Y(1)

In [ ]:
# Baseline conversion probability: function of engagement & spend
baseline_prob = (
    0.05
    + 0.10 * df['engagement_score']
    + 0.02 * np.log1p(df['past_spend']) / np.log1p(500)
).clip(0.02, 0.40)

# Y(0): control outcome
df['p0'] = baseline_prob.round(6)
df['y0'] = rng.binomial(1, df['p0'])

# Y(1): treated outcome  |  p1 = p0 + ITE, clipped to [0,1]
df['p1'] = (df['p0'] + df['ite_true']).clip(0, 1).round(6)
df['y1'] = rng.binomial(1, df['p1'])

print(f"Control conversion rate:   {df['y0'].mean():.4f}")
print(f"Treatment conversion rate: {df['y1'].mean():.4f}")
print(f"True ATE (E[Y1-Y0]):       {(df['y1'] - df['y0']).mean():.4f}")
print(f"True CATE mean (E[ITE]):   {df['ite_true'].mean():.4f}")

## 4. Randomized Treatment Assignment

In [ ]:
# Perfect 50/50 randomization — we control this because it's synthetic
df['treatment'] = rng.permutation(
    np.repeat([0, 1], [N // 2, N - N // 2])
)

# Observed outcome: only one potential outcome is observed per user
df['y_obs'] = np.where(df['treatment'] == 1, df['y1'], df['y0'])

print(f"Treatment group size: {df['treatment'].sum():,}")
print(f"Control group size:   {(df['treatment']==0).sum():,}")
print(f"\nObserved conversion (control):   {df.loc[df.treatment==0,'y_obs'].mean():.4f}")
print(f"Observed conversion (treatment): {df.loc[df.treatment==1,'y_obs'].mean():.4f}")

## 5. Segment Analysis — True ITE by Segment

In [ ]:
# Create interpretable segment labels
def label_segment(row):
    d = row['device']
    if d == 'tablet':
        return 'Tablet'
    t = row['tenure_days']
    if t < 90:
        return f'New {d.capitalize()} (< 90d)'
    elif t < 365:
        return f'Mid {d.capitalize()} (90–365d)'
    else:
        return f'Long {d.capitalize()} (> 365d)'

df['segment'] = df.apply(label_segment, axis=1)

seg_summary = (
    df.groupby('segment')
      .agg(
          n=('user_id', 'count'),
          pct=('user_id', lambda x: len(x)/N*100),
          avg_true_ite=('ite_true', 'mean'),
          conversion_control=('y0', 'mean'),
          conversion_treated=('y1', 'mean'),
      )
      .sort_values('avg_true_ite', ascending=False)
      .round(4)
)
print('\n=== True ITE by Segment ===\n')
print(seg_summary.to_string())

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Synthetic Dataset — Ground Truth Overview', fontsize=16, fontweight='bold', y=1.01)

# ── Panel 1: ITE Distribution ─────────────────────────────────────────────────
ax = axes[0]
ax.hist(df['ite_true'], bins=80, color=PALETTE[0], alpha=0.85, edgecolor='none')
ax.axvline(df['ite_true'].mean(), color=PALETTE[2], lw=2, linestyle='--',
           label=f"ATE = {df['ite_true'].mean():.4f}")
ax.set_xlabel('True ITE (conversion lift)', fontsize=12)
ax.set_ylabel('Number of users', fontsize=12)
ax.set_title('Distribution of True Treatment Effects', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.4)

# ── Panel 2: Avg ITE by Segment (horizontal bar) ──────────────────────────────
ax = axes[1]
seg_plot = seg_summary['avg_true_ite'].sort_values()
colors = [PALETTE[2] if v < 0 else PALETTE[0] for v in seg_plot.values]
bars = ax.barh(seg_plot.index, seg_plot.values * 100, color=colors, alpha=0.9)
ax.axvline(0, color='#8b949e', lw=1)
ax.set_xlabel('Average True ITE (percentage points)', fontsize=12)
ax.set_title('Average ITE by User Segment', fontsize=13)
for bar, val in zip(bars, seg_plot.values * 100):
    ax.text(val + 0.05 if val >= 0 else val - 0.05,
            bar.get_y() + bar.get_height()/2,
            f'{val:+.2f}pp', va='center',
            ha='left' if val >= 0 else 'right',
            fontsize=9, color='#e6edf3')
ax.grid(True, axis='x', alpha=0.4)

# ── Panel 3: ITE vs Engagement Score ──────────────────────────────────────────
ax = axes[2]
sample = df.sample(5000, random_state=SEED)
device_colors = {'mobile': PALETTE[0], 'desktop': PALETTE[1], 'tablet': PALETTE[2]}
for dev, grp in sample.groupby('device'):
    ax.scatter(grp['engagement_score'], grp['ite_true'] * 100,
               alpha=0.3, s=8, color=device_colors[dev], label=dev.capitalize())
ax.set_xlabel('Engagement Score', fontsize=12)
ax.set_ylabel('True ITE (pp)', fontsize=12)
ax.set_title('ITE vs Engagement Score by Device', fontsize=13)
ax.legend(markerscale=3)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('figures/01_ground_truth_overview.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print('Figure saved to figures/01_ground_truth_overview.png')

## 7. Save Dataset

In [ ]:
# Full dataset (with ground truth ITE — used for model validation)
df.to_parquet('data/synthetic_users_full.parquet', index=False)

# Blinded dataset (as if this were a real experiment — ITE columns hidden)
blind_cols = ['user_id', 'age', 'tenure_days', 'device', 'past_spend',
              'engagement_score', 'treatment', 'y_obs', 'segment']
df[blind_cols].to_parquet('data/synthetic_users_blinded.parquet', index=False)

print(f'Saved synthetic_users_full.parquet    — {len(df):,} rows, {len(df.columns)} cols')
print(f'Saved synthetic_users_blinded.parquet — {len(df):,} rows, {len(blind_cols)} cols')
print('\nColumn reference:')
print(df.dtypes)

## 8. Key Takeaways

- The dataset has **strong heterogeneity by design**: new mobile users see ~+10pp lift, tablet users see ~-1pp, long-tenure desktop users near zero.
- The average treatment effect (ATE) masks this entirely — a model that only computes ATE would recommend "ship to everyone" and waste budget on the zero-effect segments.
- In Notebook 03, we will see whether our uplift models can *recover this ranking* — that's the ground-truth validation no real-data project can perform.

**Next:** `02_ab_test_analysis.ipynb` — power analysis, SRM checks, core A/B testing.